# 🎨 一站式：MiniMax H3 文生图 + 图生视频（Kaggle T4×2）

**一次 Run All**：下载模型 → 编译 sd.cpp → 启动服务 → 打通隧道 → 测试 API。

## 为什么用这个版本
之前的方案要把 40GB 模型**上传成 Kaggle Dataset**，这一步会把内存撑爆、导致
`Kernel died unexpectedly`。这个版本**完全不碰 Dataset**：
- 模型直接下载到 `/tmp`（临时盘 2TB+，够放）；
- sd.cpp 在当前会话现编译；
- 全程本地软链，不需要 Add Data、不需要搜索、不需要上传。

> ⚠️ Kaggle 同时只能开一个 GPU 会话。用这个 notebook 前，请把聊天服务那个会话关掉。
> 加速器选 **GPU T4×2**。整个流程约 20-30 分钟。

## ① 下载模型到 /tmp（约 40GB，几分钟）

断点续传，重复运行会跳过已完成的文件。带实时进度输出。

In [ ]:
import os, sys, subprocess, time, shutil
from pathlib import Path

DL_DIR = Path("/tmp/h3_dl")
DL_DIR.mkdir(parents=True, exist_ok=True)

# (HF 仓库, 仓库内路径) —— 文件名与仓库逐一核实过
FILES = [
    ("Hearthscale/MiniMax-H3-Image-GGUF", "minimax_h3_image_pruned-Q4_K_M.gguf"),
    ("Comfy-Org/MiniMax-H3-Image", "vae/minimax_h3_image_vae_fp16.safetensors"),
    ("leejet/MiniMax-H3-GGUF", "minimax_h3_fl2va_pruned-Q4_K_M.gguf"),
    ("Comfy-Org/MiniMax-H3", "vae/minimax_h3_video_vae_fp16.safetensors"),
    ("Comfy-Org/MiniMax-H3", "vae/minimax_h3_audio_vae_fp32.safetensors"),
    ("leejet/MiniMax-H3-GGUF", "qwen3vl_32b_minimax_h3-Q2_K_M.gguf"),
]
NEED = {Path(rel).name for _, rel in FILES}

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "hf_transfer", "huggingface_hub"])

def download(repo, rel):
    from huggingface_hub import hf_hub_download
    dst = DL_DIR / Path(rel).name
    if dst.exists() and dst.stat().st_size > 1024:
        print("  ⏭  已存在，跳过: %s  (%.1f GB)" % (dst.name, dst.stat().st_size/2**30), flush=True)
        return
    print("  ⬇️  下载: %s / %s" % (repo, rel), flush=True)
    for use_ft in (True, False):
        os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1" if use_ft else "0"
        for attempt in range(3):
            try:
                path = hf_hub_download(repo_id=repo, filename=rel,
                                       local_dir=str(DL_DIR),
                                       local_dir_use_symlinks=False,
                                       resume_download=True)
                if Path(path).resolve() != dst.resolve():
                    shutil.move(str(path), str(dst))
                print("      ✅ 完成: %s  (%.1f GB)" % (dst.name, dst.stat().st_size/2**30), flush=True)
                return
            except Exception as e:
                mode = "加速" if use_ft else "标准"
                print("      ⚠️ %s下载第 %d 次失败: %s" % (mode, attempt+1, str(e)[:100]), flush=True)
                for tmp in DL_DIR.rglob("*.incomplete"):
                    try: tmp.unlink()
                    except Exception: pass
                time.sleep(3)
    raise RuntimeError("下载失败: %s" % rel)

for repo, rel in FILES:
    download(repo, rel)

# 拍平到根目录（hf 可能建 vae/ 子目录）
for p in list(DL_DIR.rglob("*")):
    if p.is_file() and p.name in NEED and p.parent != DL_DIR:
        t = DL_DIR / p.name
        if not t.exists(): shutil.move(str(p), str(t))
        else: p.unlink()
for d in sorted((x for x in DL_DIR.iterdir() if x.is_dir()), key=lambda x: -len(x.parts)):
    shutil.rmtree(str(d), ignore_errors=True)

missing = [n for n in NEED if not (DL_DIR/n).is_file()]
assert not missing, "缺文件: %s" % missing
print("\n✅ 模型全部就绪:", DL_DIR, flush=True)
total = 0
for f in sorted(DL_DIR.iterdir()):
    if f.is_file():
        g = f.stat().st_size/2**30; total += g
        print("   %-52s %.1f GB" % (f.name, g))
print("   合计: %.1f GB" % total)


## ② 编译 sd.cpp（CUDA sm_75 / T4，约 10-15 分钟）

编译输出会持续打印（这是心跳，防止 Kaggle 判定无响应）。

In [ ]:
%%bash
set -e
echo "📦 安装依赖..."
apt-get update -qq
apt-get install -qq pciutils build-essential cmake git curl libcurl4-openssl-dev
ln -sf /usr/local/nvidia/lib64/libcuda.so /usr/local/cuda/lib64/libcuda.so 2>/dev/null || true

echo "📥 克隆 sd.cpp 主仓库..."
rm -rf /tmp/sd_build
git clone --depth=1 https://github.com/leejet/stable-diffusion.cpp /tmp/sd_build
cd /tmp/sd_build

echo "🔗 初始化全部子模块（上次失败的原因：少了这一步）..."
git submodule update --init --recursive
echo "   子模块就位:"
for d in ggml gml thirdparty/cutlass thirdparty/libwebp thirdparty/libwebm; do
  [ -f "$d/CMakeLists.txt" ] && echo "   ✅ $d" || echo "   ⚠️  $d (可能不需要或路径不同)"
done

echo "⚙️  cmake 配置 (CUDA sm_75 / T4)..."
cmake -B build \
  -DSD_CUDA=ON \
  -DCMAKE_BUILD_TYPE=Release \
  -DCMAKE_CUDA_ARCHITECTURES=75 \
  -DSD_CUDA_CUTLASS_PATH:PATH= \
  -DSD_CUTLASS_PATH:PATH=

echo "🔨 编译中..."
cmake --build build -j$(nproc)
echo "✅ 编译完成"
ls -lh build/bin/sd-server build/bin/sd-cli


## ③ 组装 models 目录 + 写 recipes

把 `/tmp` 里的模型软链到 sd-server 需要的目录结构，并写好"文生图 / 图生视频"两个配方。

In [ ]:
import os, json, shutil, subprocess, time, requests
from pathlib import Path

WORK = Path("/kaggle/working")
PORT = 1234
DL_DIR = Path("/tmp/h3_dl")
MROOT = Path("/tmp/sd_models")          # model root 放 /tmp（40GB，只有这里放得下）
BIN = Path("/tmp/sd_build/build/bin")
BUILD = Path("/tmp/sd_build/build")
SDBIN = Path("/tmp/sd_run")             # 二进制 + .so 也放 /tmp

NEED = ["minimax_h3_image_pruned-Q4_K_M.gguf",
        "minimax_h3_image_vae_fp16.safetensors",
        "minimax_h3_fl2va_pruned-Q4_K_M.gguf",
        "minimax_h3_video_vae_fp16.safetensors",
        "minimax_h3_audio_vae_fp32.safetensors",
        "qwen3vl_32b_minimax_h3-Q2_K_M.gguf"]
assert all((DL_DIR/n).is_file() for n in NEED), "模型文件不齐全，回步骤①"
assert (BIN/"sd-server").exists(), "sd-server 没编译出来，回步骤②"

img_main = DL_DIR/"minimax_h3_image_pruned-Q4_K_M.gguf"
img_vae  = DL_DIR/"minimax_h3_image_vae_fp16.safetensors"
vid_main = DL_DIR/"minimax_h3_fl2va_pruned-Q4_K_M.gguf"
vid_vae  = DL_DIR/"minimax_h3_video_vae_fp16.safetensors"
aud_vae  = DL_DIR/"minimax_h3_audio_vae_fp32.safetensors"
text_enc = DL_DIR/"qwen3vl_32b_minimax_h3-Q2_K_M.gguf"
IMG_MAIN, VID_MAIN = img_main.name, vid_main.name
TE, IMG_VAE, VID_VAE, AUD_VAE = text_enc.name, img_vae.name, vid_vae.name, aud_vae.name

# 建 model root 的 category 目录。关键：用 hardlink 不用 symlink——
# sd.cpp 会拒绝指向 model 目录外的 symlink，hardlink 则是同一个文件、不占额外空间。
for sub in ("diffusion_models", "vae", "text_encoders", "clip_vision", "recipes"):
    (MROOT/sub).mkdir(parents=True, exist_ok=True)

def place(src, rel):
    dst = MROOT/rel
    if dst.is_symlink() or (dst.exists() and dst.stat().st_size == 0): dst.unlink()
    if dst.exists(): return
    try:
        os.link(str(src), str(dst)); print("   hardlink", rel)
    except OSError:
        shutil.copy2(str(src), str(dst)); print("   copy    ", rel)

place(img_main, f"diffusion_models/{IMG_MAIN}")
place(vid_main, f"diffusion_models/{VID_MAIN}")
place(text_enc, f"text_encoders/{TE}")
place(img_vae,  f"vae/{IMG_VAE}")
place(vid_vae,  f"vae/{VID_VAE}")
place(aud_vae,  f"vae/{AUD_VAE}")

# 关键：不要放自定义 recipe！内置 recipe 已编译进二进制（minimax-h3-image / minimax-h3-fl2va）。
# 清掉旧版写错的 recipe，否则会覆盖内置、导致 unknown recipe。
for f in (MROOT/"recipes").glob("*.json"):
    f.unlink()

# 二进制 + .so 复制到可写目录
SDBIN.mkdir(parents=True, exist_ok=True)
for so in BUILD.rglob("*.so*"):
    if "CMakeFiles" not in str(so) and so.is_file():
        d = SDBIN/so.name
        if not d.exists(): shutil.copy2(str(so), str(d))
server_dst = SDBIN/"sd-server"
shutil.copy2(str(BIN/"sd-server"), str(server_dst))
server_dst.chmod(0o755)
os.environ["LD_LIBRARY_PATH"] = str(SDBIN) + ":" + os.environ.get("LD_LIBRARY_PATH","")

MODELS = MROOT
print("✅ model root:", MODELS)
print("   可见类别:", sorted(p.name for p in MODELS.iterdir() if p.is_dir()))


## ④ 启动 sd-server（后台常驻，0.0.0.0:1234）

In [ ]:
import subprocess, os, time, requests
subprocess.run(["pkill", "-9", "-f", "sd-server"], capture_output=True)
time.sleep(2)
log = open("/tmp/sd-server.log", "w")
cmd = [str(server_dst), "--model-dir", str(MODELS),
       "--listen-ip", "0.0.0.0", "--listen-port", str(PORT), "--offload-to-cpu"]
env = {**os.environ, "LD_LIBRARY_PATH": str(SDBIN)+":"+os.environ.get("LD_LIBRARY_PATH","")}
proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=env,
                        start_new_session=True)
print("sd-server 启动中 (pid=%d)..." % proc.pid)
for i in range(90):
    time.sleep(2)
    try:
        r = requests.get(f"http://127.0.0.1:{PORT}/sdcpp/v1/models/status", timeout=3)
        print("✅ sd-server 就绪:", r.status_code, r.text[:160]); break
    except Exception:
        if proc.poll() is not None:
            print("❌ 进程退出，日志:"); print(open("/tmp/sd-server.log").read()[-2000:]); break
else:
    print("⏳ 仍在启动，日志尾部:"); print(open("/tmp/sd-server.log").read()[-1200:])

# 列出服务器实际识别到的文件 ID 和 recipe（加载时必须用这里的 id）
BASE = f"http://127.0.0.1:{PORT}"
mf = requests.get(BASE+"/sdcpp/v1/model-files", timeout=10).json()
print("\n📁 可见文件 ID:")
for f in mf.get("files", []): print("   ", f["id"], "  [", f["category"], "]")
rc = requests.get(BASE+"/sdcpp/v1/recipes", timeout=10).json()
h3 = [r["id"] for r in rc.get("recipes", []) if "h3" in r["id"].lower()]
print("\n📋 MiniMax-H3 内置 recipes:", h3)


## ⑤ 打通 cloudflared 隧道（拿到公网 API 地址）

In [ ]:
import subprocess, time, requests, os

# Kaggle 默认没装 cloudflared，缺失则自动安装（与聊天服务一致）
import shutil
if shutil.which("cloudflared") is None:
    print("📦 安装 cloudflared ...", flush=True)
    r = subprocess.run(["wget","-q",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-O","/usr/local/bin/cloudflared"], capture_output=True, text=True)
    subprocess.run(["chmod","+x","/usr/local/bin/cloudflared"])
    if shutil.which("cloudflared") is None:
        raise RuntimeError("cloudflared 安装失败: " + (r.stderr or "")[-500:])

subprocess.run(["pkill", "-9", "-f", "cloudflared"], capture_output=True)
time.sleep(3)
clog = "/tmp/cloudflare.log"
with open(clog, "w") as f:
    subprocess.Popen(["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}"],
                     stdout=f, stderr=subprocess.STDOUT, start_new_session=True)

def find_url(txt):
    for tok in txt.replace("|", " ").split():
        if "https://" in tok and "trycloudflare.com" in tok:
            i = tok.find("https://"); j = tok.find("trycloudflare.com", i)
            return tok[i:j+len("trycloudflare.com")]
    return None

url = None
for _ in range(45):
    time.sleep(2)
    url = find_url(open(clog, errors="ignore").read())
    if url: break
print("="*60)
if url:
    print("✅ API 地址 (BASE):", url)
    print("   网页界面:", url + "/")
else:
    print("❌ 隧道未拿到地址，日志尾部:")
    print(open(clog, errors="ignore").read()[-1500:])
print("="*60)


## ⑥ 测试 API（文生图 + 图生视频）

用 localhost 测，不受隧道 100 秒超时影响。首次加载模型 1-2 分钟正常，图生视频在 T4 上约几分钟。

In [ ]:
import requests, json, time, base64
BASE = f"http://127.0.0.1:{PORT}"

def load_model(recipe, models):
    print(f"⏳ 加载 {recipe}（同步等待，通常 2-5 分钟）...", flush=True)
    r = requests.post(BASE+"/sdcpp/v1/models/load",
                      json={"recipe": recipe, "models": models}, timeout=(10, 900))
    print("   load:", r.status_code, r.text[:400])
    return r

def gen_and_wait(endpoint, body, outfile, timeout=1800):
    r = requests.post(BASE+endpoint, json=body, timeout=60)
    print("   submit:", r.status_code, r.text[:200])
    if r.status_code != 202:
        print("   ❌ 提交失败"); return None
    jid = r.json()["id"]
    t0 = time.time()
    while time.time()-t0 < timeout:
        time.sleep(5)
        j = requests.get(BASE+f"/sdcpp/v1/jobs/{jid}", timeout=30).json()
        st = j.get("status")
        pr = j.get("progress") or {}
        if st in ("queued", "generating", "running"):
            print(f"   [{int(time.time()-t0)}s] {st} stage={pr.get('stage')} "
                  f"{pr.get('completed')}/{pr.get('total')} {pr.get('unit')}")
            continue
        if st == "completed":
            res = j.get("result") or {}
            b64 = res.get("b64_json")                      # vid_gen
            if not b64:
                imgs = res.get("images") or []             # img_gen
                if imgs: b64 = imgs[0].get("b64_json")
            if b64:
                if b64.startswith("data:") and "," in b64[:120]:
                    b64 = b64.split(",", 1)[1]
                data = base64.b64decode(b64)
                open(outfile, "wb").write(data)
                print(f"   ✅ 保存 {outfile} ({len(data)/1024/1024:.1f} MB) "
                      f"耗时 {int(time.time()-t0)}s")
            else:
                print("   ⚠️ completed 但无 b64:", str(res)[:500])
            return j
        print("   ❌ 失败:", str(j.get("error"))[:400]); return None
    print("   ⏳ 超时"); return None

# A. 文生图 —— 内置 recipe + model-files 返回的真实 file ID
load_model("minimax-h3-image", {
    "diffusion_model": "diffusion_models/minimax_h3_image_pruned-Q4_K_M.gguf",
    "llm":            "text_encoders/qwen3vl_32b_minimax_h3-Q2_K_M.gguf",
    "vae":            "vae/minimax_h3_image_vae_fp16.safetensors",
})
gen_and_wait("/sdcpp/v1/img_gen", {
    "prompt": "A silver tabby kitten holding a sign reading 'hi', natural daylight, photorealistic",
    "width": 1024, "height": 1024, "seed": 42, "output_format": "png"
}, "/kaggle/working/test_image.png", timeout=1800)
print("\n✅ 文生图测试完成\n")


In [ ]:
# B. 图生视频（低负载版：自动重启 sd-server + 无声 22 帧，避免 OOM）
import requests, base64, os, time, subprocess
from pathlib import Path
BASE = f"http://127.0.0.1:{PORT}"
MROOT = Path("/tmp/sd_models"); SDBIN = Path("/tmp/sd_run")
server_dst = SDBIN / "sd-server"

# 0) 先看上次崩溃日志尾部（诊断：找 CUDA out of memory / bad_alloc）
print("=== 上次崩溃日志尾部（诊断）===")
try: print(open("/tmp/sd-server.log", errors="ignore").read()[-1000:])
except Exception as e: print("(无日志)", e)

# 1) 重启 sd-server
print("\n=== 重启 sd-server ===")
subprocess.run(["pkill","-9","-f","sd-server"], capture_output=True); time.sleep(3)
env = {**os.environ, "LD_LIBRARY_PATH": str(SDBIN)+":"+os.environ.get("LD_LIBRARY_PATH","")}
log = open("/tmp/sd-server.log","w")
proc = subprocess.Popen([str(server_dst),"--model-dir",str(MROOT),
    "--listen-ip","0.0.0.0","--listen-port",str(PORT),"--offload-to-cpu"],
    stdout=log, stderr=subprocess.STDOUT, env=env, start_new_session=True)
for i in range(90):
    time.sleep(2)
    try:
        r = requests.get(BASE+"/sdcpp/v1/models/status", timeout=3)
        print("sd-server 就绪:", r.status_code); break
    except Exception:
        if proc.poll() is not None:
            print("进程退出，日志:"); print(open("/tmp/sd-server.log").read()[-2000:]); break
else:
    print("启动超时，日志:"); print(open("/tmp/sd-server.log").read()[-1200:])

def gen_and_wait(endpoint, body, outfile, timeout=1800):
    r = requests.post(BASE+endpoint, json=body, timeout=60)
    print("submit:", r.status_code, r.text[:200])
    if r.status_code != 202: return None
    jid = r.json()["id"]; t0 = time.time()
    while time.time()-t0 < timeout:
        time.sleep(5)
        j = requests.get(BASE+f"/sdcpp/v1/jobs/{jid}", timeout=30).json()
        st = j.get("status"); pr = j.get("progress") or {}
        if st in ("queued","generating","running"):
            print(f"[{int(time.time()-t0)}s] {st} {pr.get('stage')} {pr.get('completed')}/{pr.get('total')}"); continue
        if st == "completed":
            res = j.get("result") or {}
            b64 = res.get("b64_json")
            if not b64:
                imgs = res.get("images") or []
                if imgs: b64 = imgs[0].get("b64_json")
            if b64:
                data = base64.b64decode(b64.split(",",1)[-1] if b64.startswith("data:") else b64)
                open(outfile,"wb").write(data)
                print(f"✅ 保存 {outfile} ({len(data)/1024/1024:.1f} MB)")
            return j
        print("❌ failed:", str(j.get("error"))[:400]); return None
    print("超时"); return None

# 2) 加载视频模型（无 audio_vae → 先出无声视频，省显存）
print("\n=== 加载 fl2va（无音频 VAE）===")
r = requests.post(BASE+"/sdcpp/v1/models/load", json={
    "recipe": "minimax-h3-fl2va",
    "models": {
        "diffusion_model": "diffusion_models/minimax_h3_fl2va_pruned-Q4_K_M.gguf",
        "llm": "text_encoders/qwen3vl_32b_minimax_h3-Q2_K_M.gguf",
        "vae": "vae/minimax_h3_video_vae_fp16.safetensors",
    }
}, timeout=(10,900))
print("load:", r.status_code, r.text[:400])

# 3) 低负载视频：640x384、22 帧（=17*1+5，符合 frame_step=17/offset=5）
if r.status_code == 200:
    first = "/kaggle/working/test_image.png"
    assert os.path.exists(first), "找不到 test_image.png"
    b64img = "data:image/png;base64," + base64.b64encode(open(first,"rb").read()).decode()
    print("\n=== 生成视频（640x384, 22 帧, 无声）===")
    gen_and_wait("/sdcpp/v1/vid_gen", {
        "prompt": "the kitten gently turns its head and blinks, subtle motion, natural light",
        "width": 640, "height": 384, "video_frames": 22, "fps": 24, "seed": 42,
        "init_image": b64img, "output_format": "webm"
    }, "/kaggle/working/test_video.webm", timeout=1800)
    print("\n✅ 图生视频测试结束")
else:
    print("⚠️ 视频模型加载失败，未提交视频任务——把上面 load 输出发我")


## ✅ 完成后

拿到 `test_image.png` 和 `test_video.webm` 就说明全链路通了。

**API 调用**（Agent / 程序）用步骤⑤打印的 `BASE` 地址：
- `POST /sdcpp/v1/img_gen` — 文生图（图生图把 `init_image` 设为图片 base64）
- `POST /sdcpp/v1/vid_gen` — 图生视频（`init_image`=首帧图，`video_frames` 帧数）
- `POST /sdcpp/v1/models/load` — 切换文生图/视频模型
- `GET /sdcpp/v1/jobs/{id}` — 轮询取结果

⚠️ 隧道 100 秒超时：出图约 60s 勉强可用；出视频必超时，程序化调用请走 notebook 内 `127.0.0.1:1234` 或用异步轮询。

按你的规矩：确认跑通后我才推 GitHub。现在仓库未动。